# The Creative Prisim
## Phase 1 — Full Studio Workflow
**The Intelligence of Seeing (IOS) — v1.3**

Nine-stage workflow with Studio Brief Document threading.
Corrected role sequence: Creator → Critic → Researcher.

**v1.3 fixes for Experiment 3:**
- Gap 1: Critic max_tokens raised 1200 → 1800
- Gap 2: Director review converted to JSON format with validation gate
- Gap 3: Researcher task rewritten — sourcing language, citation requirement, silence permission
- Gap 4: Stage 8b added — Director depth extraction before final synthesis
- Gap 5: Iteration loop added — re-runs full Creative Team when Director signals ITERATE
- Python validation layer: validate_stage() calls at each stage gate

```
Stage 1   Discovery
Stage 2   Reframing
Stage 3   Creative Brief  ← JSON format, validation gate
Stage 4a  Creator         ← brief goes here first
Stage 4b  Critic          ← evaluates Creator output
Stage 4c  Researcher      ← sources external precedents post-ideation
Stage 5   Candidate Directions
Stage 6   Director Review ← JSON format, validation gate
Stage 6a  Iteration Loop  ← re-runs full Creative Team if ITERATE
Stage 7   Presentation
Stage 8   User Reaction
Stage 8b  Depth Extraction ← Director probes deeper before synthesis
Stage 9   Final Synthesis
```

---
## Cell 1 — Load Phase 0 Foundation

In [ ]:
# Import Creative Prisim engine
# All infrastructure lives in engine.py — clean, no re-execution overhead
import json

from engine import (
    client,
    create_blackboard,
    scribe_log,
    build_prompt,
    call_role,
    save_session,
    verify_engine,
    create_brief_doc,
    update_brief_doc,
    read_brief_doc,
    ROLE_WEIGHTS,
    GLOBAL_MODES,
    apply_global_mode,
    DEFAULT_MODEL,
    validate_stage
)

engine_ready = verify_engine()
if not engine_ready:
    raise RuntimeError("Engine verification failed. Fix issues before running session.")

print("\n✓ Engine imported and verified — ready for Phase 1 v1.3")

---
## Cell 2 — Session Configuration

Set `TEST_MODE = True` for automated pipeline testing.
Set `TEST_MODE = False` for live conversation with the PIL.

In [ ]:
# ── SESSION CONFIGURATION ─────────────────────────────────────────────────────

TEST_MODE = True   # ← Set to False for live PIL interaction

# ── TEST BRIEF (used when TEST_MODE = True) ────────────────────────────────────
# A rich brief that will exercise all roles meaningfully.
# This is the bookstore café problem from Phase 0, now fully specified.

TEST_BRIEF = {
    "initial_prompt": "I need to develop a brand identity for a new independent bookstore café.",
    "context": "Small urban space in a gentrifying neighborhood. The owner is a former librarian "
               "who wants to create a community gathering place, not just a retail experience. "
               "Opening in six months. Budget is modest.",
    "audience": "Local residents, remote workers, students, book lovers. Ages 25-55. "
                "People who are slightly skeptical of corporate coffee culture.",
    "desired_outcome": "A brand that feels genuinely rooted in the neighborhood — "
                       "intellectual without being intimidating, warm without being generic.",
    "preferences": ["feels earned, not designed", "has a point of view", "avoids clichés"],
    "constraints": ["modest budget", "must work as signage, social media, and packaging"],
    "what_to_avoid": "Anything that feels like a chain trying to look independent. "
                     "No Edison bulbs. No 'artisanal' language."
}

# ── MODEL CONFIGURATION ────────────────────────────────────────────────────────
# Haiku for development (fast, cheap)
# Swap to claude-sonnet-4-6 for quality demo runs

SESSION_MODEL = "claude-haiku-4-5-20251001"
SESSION_MODE  = "balanced"   # exploratory | balanced | execution

print(f"✓ Configuration loaded")
print(f"  TEST_MODE     : {TEST_MODE}")
print(f"  Model         : {SESSION_MODEL}")
print(f"  Global mode   : {SESSION_MODE}")
if TEST_MODE:
    print(f"  Brief topic   : {TEST_BRIEF['initial_prompt']}")

---
## Cell 3 — Stage 1: Discovery

The Director gathers context from the PIL.
In TEST_MODE: brief is pre-loaded.
In live mode: Director conducts a conversational discovery exchange.

In [ ]:
# ── STAGE 1: DISCOVERY ────────────────────────────────────────────────────────

print("═" * 60)
print("STAGE 1 — DISCOVERY")
print("═" * 60)

if TEST_MODE:
    # ── Test Mode: use pre-loaded brief ───────────────────────────────────────
    print("[TEST MODE — pre-loaded brief]\n")

    initial_prompt = TEST_BRIEF["initial_prompt"]
    blackboard = create_blackboard(initial_prompt, system_version="1.2")
    scribe_log(blackboard, "SYSTEM", "session_start",
               f"TEST MODE session initiated: '{initial_prompt}'")

    # Build discovery record from test brief
    blackboard["discovery"]["orientation_summary"] = initial_prompt
    blackboard["discovery"]["context_insights"]    = [
        TEST_BRIEF["context"],
        TEST_BRIEF["audience"]
    ]
    blackboard["discovery"]["desired_outcome"]     = TEST_BRIEF["desired_outcome"]
    blackboard["discovery"]["preferences"]         = TEST_BRIEF["preferences"]
    blackboard["discovery"]["sacrificial_exploration"]["probe_prompt"] = \
        "What if this place behaved more like a private library than a café?"
    blackboard["discovery"]["sacrificial_exploration"]["user_response"] = \
        "Actually that's interesting — the owner does want it to feel more like " \
        "a place you belong to than a place you visit."
    blackboard["discovery"]["sacrificial_exploration"]["insight_extracted"] = \
        "Membership/belonging framing may be stronger than customer/visitor framing."

    scribe_log(blackboard, "DIRECTOR", "discovery_complete",
               "Discovery loaded from test brief. Belonging/membership signal extracted.")

    print(f"Initial prompt : {initial_prompt}")
    print(f"Context        : {TEST_BRIEF['context'][:80]}...")
    print(f"Desired outcome: {TEST_BRIEF['desired_outcome'][:80]}...")
    print(f"Key signal     : {blackboard['discovery']['sacrificial_exploration']['insight_extracted']}")

else:
    # ── Live Mode: Director conducts discovery conversation ────────────────────
    print("[LIVE MODE — PIL interaction]\n")

    initial_prompt = input("What would you like to explore today?\n> ").strip()
    blackboard = create_blackboard(initial_prompt, system_version="1.2")
    scribe_log(blackboard, "SYSTEM", "session_start",
               f"LIVE session initiated: '{initial_prompt}'")

    # Director conducts discovery via API
    discovery_task = f"""The person has brought this challenge:
'{initial_prompt}'

Conduct a brief discovery conversation. Ask ONE clear, open question to
understand context, audience, or desired outcome. Keep it conversational.
Do not ask multiple questions at once."""

    director_q1 = call_role(
        role="director",
        user_message=discovery_task,
        weights=ROLE_WEIGHTS["director"],
        context=initial_prompt,
        blackboard=blackboard,
        model=SESSION_MODEL
    )
    print(f"\nDirector: {director_q1}\n")
    pil_response_1 = input("> ").strip()

    # One follow-up question
    followup_task = f"""Discovery so far:
Prompt: {initial_prompt}
Context gathered: {pil_response_1}

Ask ONE more focused question — about desired outcome or what to avoid.
Then stop. Do not ask more than two questions total."""

    director_q2 = call_role(
        role="director",
        user_message=followup_task,
        weights=ROLE_WEIGHTS["director"],
        context=initial_prompt,
        blackboard=blackboard,
        model=SESSION_MODEL
    )
    print(f"\nDirector: {director_q2}\n")
    pil_response_2 = input("> ").strip()

    # Store discovery
    blackboard["discovery"]["orientation_summary"] = initial_prompt
    blackboard["discovery"]["context_insights"]    = [pil_response_1]
    blackboard["discovery"]["desired_outcome"]     = pil_response_2

    scribe_log(blackboard, "DIRECTOR", "discovery_complete",
               f"Two-exchange discovery complete. Context and outcome captured.")
    print("\n✓ Discovery complete")

# ── Create Studio Brief Document ──────────────────────────────────────────────
session_id    = blackboard["session_metadata"]["session_id"]
brief_doc_path = create_brief_doc(session_id, blackboard["user_prompt"])

discovery_section = f"""**Initial prompt:** {blackboard['user_prompt']}
**Context:** {" | ".join(blackboard['discovery']['context_insights'])}
**Desired outcome:** {blackboard['discovery']['desired_outcome']}
**Key signal:** {blackboard['discovery']['sacrificial_exploration']['insight_extracted']}"""

update_brief_doc(session_id, "DIRECTOR", "DISCOVERY SUMMARY", discovery_section)

print(f"\n✓ Blackboard discovery populated")
print(f"✓ Studio Brief Document created: brief_{session_id[:8]}...")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 4 — Stage 2 & 3: Reframing + Creative Brief

The Director synthesizes discovery into a reframed challenge and structured brief.
This becomes the working document for the Creative Team.

In [ ]:
# ── STAGE 2: REFRAMING ────────────────────────────────────────────────────────
# ── STAGE 3: CREATIVE BRIEF ──────────────────────────────────────────────────

print("═" * 60)
print("STAGE 2 — REFRAMING")
print("STAGE 3 — CREATIVE BRIEF")
print("═" * 60)

# Build the discovery summary for the Director
discovery = blackboard["discovery"]
discovery_summary = f"""Initial prompt: {blackboard['user_prompt']}
Context insights: {" | ".join(discovery['context_insights'])}
Desired outcome: {discovery['desired_outcome']}
Preferences: {", ".join(discovery['preferences']) if discovery['preferences'] else "not specified"}
Sacrificial probe insight: {discovery['sacrificial_exploration']['insight_extracted']}"""

if TEST_MODE:
    discovery_summary += f"""
Constraints: {", ".join(TEST_BRIEF['constraints'])}
What to avoid: {TEST_BRIEF['what_to_avoid']}"""

# Director synthesizes brief — JSON format for reliable parsing
brief_task = f"""Based on this discovery:

{discovery_summary}

Produce a structured creative brief as a JSON object. Return ONLY the JSON — no
preamble, no explanation, no markdown fences. Exact format:

{{
  "challenge": "one clear sentence",
  "context": "2-3 sentences about situation and audience",
  "desired_result": "what success looks like",
  "constraints": ["limit 1", "limit 2"],
  "open_questions": ["question 1", "question 2"]
}}

Be specific. This brief becomes the Creative Team's working document."""

brief_response = call_role(
    role="director",
    user_message=brief_task,
    weights=ROLE_WEIGHTS["director"],
    context=blackboard["user_prompt"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=600,
    brief_doc=read_brief_doc(session_id)
)

# ── Parse brief — JSON with validation gate ───────────────────────────────────
brief = blackboard["creative_brief"]

try:
    # Strip markdown fences if model added them despite instructions
    clean = brief_response.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    parsed = json.loads(clean.strip())

    brief["challenge"]      = parsed.get("challenge", "")
    brief["context"]        = parsed.get("context", "")
    brief["desired_result"] = parsed.get("desired_result", "")
    brief["constraints"]    = parsed.get("constraints", [])
    brief["open_questions"] = parsed.get("open_questions", [])

    # ── Validation gate ───────────────────────────────────────────────────────
    missing = [f for f in ["challenge", "context", "desired_result"]
               if not brief.get(f, "").strip()]
    if missing:
        raise ValueError(f"Brief missing required fields: {missing}")

    print("✓ Brief parsed successfully")

except (json.JSONDecodeError, ValueError) as e:
    print(f"✗ Brief parsing failed: {e}")
    print("Raw response:")
    print(brief_response)
    raise RuntimeError(
        "Brief parsing failed — session cannot continue with an incomplete brief.\n"
        "Check the Director prompt or model output above."
    )

# ── Update Studio Brief Document ──────────────────────────────────────────────
brief_section = f"""**CHALLENGE:** {brief['challenge']}

**CONTEXT:** {brief['context']}

**DESIRED RESULT:** {brief['desired_result']}

**CONSTRAINTS:** {", ".join(brief['constraints'])}

**OPEN QUESTIONS:** {" | ".join(brief['open_questions'])}"""

update_brief_doc(session_id, "DIRECTOR", "THE BRIEF", brief_section)

scribe_log(blackboard, "DIRECTOR", "brief_synthesized",
           f"Creative brief produced. Challenge: '{brief['challenge'][:60]}...'")

print("── CREATIVE BRIEF ──────────────────────────────────────────")
print(brief_response)
print("────────────────────────────────────────────────────────────")

if not TEST_MODE:
    print("\nDirector confirms reframing with PIL...")
    print(f"\nDoes this capture what you're after? (yes / adjust)")
    confirmation = input("> ").strip().lower()
    if "adjust" in confirmation or "no" in confirmation:
        adjustment = input("What should change? ").strip()
        brief["open_questions"].append(f"PIL adjustment: {adjustment}")
        scribe_log(blackboard, "DIRECTOR", "brief_adjusted",
                   f"PIL requested adjustment: '{adjustment}'")
        print("✓ Adjustment noted. Proceeding with updated brief.")

validate_stage(blackboard, "brief")

print(f"\n✓ Brief complete")
print(f"✓ Studio Brief Document updated")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 5 — Stage 4a: Creator

The creative brief goes directly to the Creator first — not the Researcher.
The Creator generates candidate directions informed by the brief.
Verbalized Sampling applied: probability weights push toward lateral thinking.
The Researcher has not yet acted — the Creative Team works first.

In [ ]:
# ── STAGE 4b: CREATOR — IDEATION ──────────────────────────────────────────────

print("═" * 60)
print("STAGE 4b — CREATOR: IDEATION")
print("═" * 60)

research_summary = blackboard["research_trace"][0]["summary"] \
    if blackboard["research_trace"] else "No research findings available."

creator_task = f"""You are working on this creative brief:

CHALLENGE: {brief['challenge']}
CONTEXT: {brief['context']}
DESIRED RESULT: {brief['desired_result']}
CONSTRAINTS: {", ".join(brief['constraints'])}

RESEARCH INSIGHTS:
{research_summary}

Generate 4 distinct conceptual directions. Use Verbalized Sampling:
for each direction, include a probability (0.0-1.0) representing how
likely this direction would be from a conventional approach.
Lower probability = more lateral, unexpected, less obvious.
Include at least one direction with probability below 0.15.

For each direction provide:
IDEA_ID: [A1, A2, A3, A4]
TITLE: [short evocative name]
PROBABILITY: [0.0-1.0]
CONCEPT: [2-3 sentences — what is this direction?]
RATIONALE: [why this is worth exploring]
EMOTIONAL TERRITORY: [what feeling does this occupy?]

Be genuinely diverse — not four variations of the same idea."""

creator_response = call_role(
    role="creator",
    user_message=creator_task,
    weights=ROLE_WEIGHTS["creator"],
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=1200,
    brief_doc=read_brief_doc(session_id)
)

# Store ideation cycle
cycle_1 = {
    "cycle_number": 1,
    "creator_proposals": [{"raw_response": creator_response}],
    "critic_feedback": [],
    "synthesized_directions": []
}
blackboard["ideation_cycles"].append(cycle_1)

# Update Studio Brief Document
update_brief_doc(session_id, "CREATOR", "DIRECTIONS EXPLORED", creator_response)

scribe_log(blackboard, "CREATOR", "ideation_complete",
           "Four directions generated using Verbalized Sampling. Cycle 1 logged.")

print("── CREATOR DIRECTIONS ──────────────────────────────────────")
print(creator_response)
print("────────────────────────────────────────────────────────────")
validate_stage(blackboard, "ideation")

print(f"\n✓ Creator ideation complete")
print(f"✓ Studio Brief Document updated")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 6 — Stage 4b: Critic

The Critic evaluates Creator directions and proposes a synthesis.
Reads the full Studio Brief Document including Creator output.
Constructive pressure — not rejection.

In [ ]:
# ── STAGE 4c: CRITIC — EVALUATION ────────────────────────────────────────────

print("═" * 60)
print("STAGE 4c — CRITIC: EVALUATION")
print("═" * 60)

creator_output = blackboard["ideation_cycles"][0]["creator_proposals"][0]["raw_response"]

critic_task = f"""The Creator has proposed these directions for the brief:

BRIEF CHALLENGE: {brief['challenge']}
DESIRED RESULT: {brief['desired_result']}

CREATOR'S DIRECTIONS:
{creator_output}

Evaluate each direction. For each:
IDEA_ID: [matching the Creator's ID]
STRENGTHS: [what genuinely works — be specific]
WEAKNESSES: [what is underdeveloped, risky, or misaligned with brief]
REFINEMENT: [one concrete suggestion to strengthen it]

Then propose ONE synthesis direction that combines the strongest elements
across multiple ideas into something the Creator didn't quite reach:
SYNTHESIS_ID: S1
TITLE: [name]
CONCEPT: [2-3 sentences]
ORIGIN_IDEAS: [which Creator ideas it draws from]

Be analytically rigorous. Your job is to strengthen, not approve."""

critic_response = call_role(
    role="critic",
    user_message=critic_task,
    weights=ROLE_WEIGHTS["critic"],
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=1800,
    brief_doc=read_brief_doc(session_id)
)

# Store critic feedback in cycle
blackboard["ideation_cycles"][0]["critic_feedback"].append({
    "raw_response": critic_response
})

# Update Studio Brief Document
update_brief_doc(session_id, "CRITIC", "CRITIQUE & SYNTHESIS", critic_response)

scribe_log(blackboard, "CRITIC", "evaluation_complete",
           "Creator directions evaluated. Synthesis direction proposed.")

print("── CRITIC EVALUATION ───────────────────────────────────────")
print(critic_response)
print("────────────────────────────────────────────────────────────")
validate_stage(blackboard, "critic")

print(f"\n✓ Critic evaluation complete")
print(f"✓ Studio Brief Document updated")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 7 — Stage 4c: Researcher

The Researcher acts AFTER the Creative Team — not before.
It reads the full Studio Brief Document including the ideation cycle
and provides contextual enrichment informed by what the team actually produced.
This is also when the bounded autonomous contribution trigger is evaluated:
the Researcher now has sufficient thinking on the Blackboard to identify genuine gaps.

In [ ]:
# ── STAGE 4c: RESEARCHER (post-ideation) ──────────────────────────────────────

print("═" * 60)
print("STAGE 4c — RESEARCHER (post-ideation)")
print("═" * 60)

brief         = blackboard["creative_brief"]
creator_output = blackboard["ideation_cycles"][0]["creator_proposals"][0]["raw_response"]
critic_output  = blackboard["ideation_cycles"][0]["critic_feedback"][0]["raw_response"]

research_task = f"""You are supporting a creative studio session.

THE BRIEF:
Challenge: {brief['challenge']}
Context: {brief['context']}
Desired result: {brief['desired_result']}

THE CREATIVE TEAM HAS PRODUCED:

CREATOR\'S DIRECTIONS:
{creator_output}

CRITIC\'S EVALUATION AND SYNTHESIS:
{critic_output}

Your task: source and cite 2-3 external precedents, examples, or factual findings
from outside this problem\'s domain that could ground or expand what the Creative Team
has produced. Respond to the actual directions above — not just the brief in isolation.

For each finding:
TOPIC: [what area, domain, or precedent]
SOURCE: [name the specific source, study, institution, or example — be precise.
         Not \'research suggests\' but \'the X study / Y institution / Z historical case\']
FINDING: [what it shows]
RELEVANCE: [why this matters specifically for these directions — be precise]

If the Creative Team\'s work is already well-grounded and you have no external
knowledge that would genuinely add to it, say only:
RESEARCHER: No external contribution warranted for this cycle.

You are a sourcing agent, not an analyst. Do not evaluate or critique the
Creative Team\'s directions — that is the Critic\'s role. Every finding must
name a specific source."""

research_response = call_role(
    role="researcher",
    user_message=research_task,
    weights=ROLE_WEIGHTS["researcher"],
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=700,
    brief_doc=read_brief_doc(session_id)
)

# Parse and store research findings — full content, no truncation
research_entry = {
    "research_id": f"R{len(blackboard['research_trace']) + 1:03d}",
    "initiated_by": "director",
    "query": "Contextual precedents and adjacent-field insights informed by ideation cycle",
    "sources": ["researcher_knowledge"],
    "summary": research_response,
    "influence_on_ideation": "Post-ideation enrichment — Researcher read Creative Team output"
}
blackboard["research_trace"].append(research_entry)
brief["research_insights"].append(research_response)

# Update Studio Brief Document
update_brief_doc(session_id, "RESEARCHER", "RESEARCH CONTEXT", research_response)

scribe_log(blackboard, "RESEARCHER", "research_delivered",
           f"Post-ideation research provided. Entry R{len(blackboard['research_trace']):03d} logged.")

print("── RESEARCHER FINDINGS ─────────────────────────────────────")
print(research_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Research logged to Blackboard")
print(f"✓ Studio Brief Document updated")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 8 — Stage 5 & 6: Candidate Set + Director Review

Director assembles candidate directions from the ideation cycle,
evaluates them against the brief, and decides whether to present
or send back for another round.

In [ ]:
# ── STAGE 5: CANDIDATE DIRECTIONS ────────────────────────────────────────────
# ── STAGE 6: DIRECTOR REVIEW ─────────────────────────────────────────────────

print("═" * 60)
print("STAGE 5 — CANDIDATE DIRECTIONS")
print("STAGE 6 — DIRECTOR REVIEW")
print("═" * 60)

creator_output = blackboard["ideation_cycles"][0]["creator_proposals"][0]["raw_response"]
critic_output  = blackboard["ideation_cycles"][0]["critic_feedback"][0]["raw_response"]

review_task = f"""You are reviewing the studio\'s work before presenting to the PIL.

BRIEF:
Challenge: {brief['challenge']}
Desired result: {brief['desired_result']}
Constraints: {", ".join(brief['constraints'])}

CREATOR DIRECTIONS:
{creator_output[:800]}

CRITIC EVALUATION:
{critic_output[:800]}

Your task:
1. Select 3 directions to present — the strongest, most distinct options.
   Include the Critic\'s synthesis direction if it is stronger than any Creator direction.
2. Evaluate the candidate set against the brief.
3. Decide: PRESENT or ITERATE.

Return ONLY a JSON object — no preamble, no explanation, no markdown fences:

{{
  "candidates": [
    {{"id": "C1", "title": "exact direction name", "type": "evolutionary|evolutionary_variant|conceptual_leap|sacrificial_probe", "source_idea_id": "A1"}},
    {{"id": "C2", "title": "...", "type": "...", "source_idea_id": "A3"}},
    {{"id": "C3", "title": "...", "type": "...", "source_idea_id": "S1"}}
  ],
  "alignment": "one sentence — does each direction address the actual challenge?",
  "distinctiveness": "one sentence — are these genuinely different approaches?",
  "balance": "one sentence — does the set span practical to exploratory?",
  "clarity": "one sentence — is each clearly expressible to the PIL?",
  "decision": "PRESENT",
  "reason": "one sentence"
}}

For ITERATE: set decision to "ITERATE" and explain why in the reason field."""

review_response = call_role(
    role="director",
    user_message=review_task,
    weights=ROLE_WEIGHTS["director"],
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=800,
    brief_doc=read_brief_doc(session_id)
)

# ── Parse Director review — JSON with validation gate ─────────────────────────
review = blackboard["director_review"]

try:
    clean = review_response.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    parsed = json.loads(clean.strip())

    review["alignment_with_brief"]       = parsed.get("alignment", "")
    review["distinctiveness_assessment"] = parsed.get("distinctiveness", "")
    review["balance_assessment"]         = parsed.get("balance", "")
    review["clarity_assessment"]         = parsed.get("clarity", "")
    decision = parsed.get("decision", "PRESENT")
    review["iteration_required"] = "ITERATE" in decision.upper()

    candidates = parsed.get("candidates", [])
    if not candidates:
        raise ValueError("Director review returned no candidates")

    for c in candidates:
        blackboard["candidate_set"].append({
            "direction_id":      c.get("id", f"D{len(blackboard['candidate_set'])+1:03d}"),
            "internal_type":     c.get("type", "evolutionary").lower(),
            "description":       c.get("title", ""),
            "reasoning_summary": "",
            "research_influences": [],
            "critic_assessment": {"strengths": [], "concerns": [], "refinement_notes": []}
        })

    print("✓ Director review parsed successfully")
    print(f"  Decision   : {decision}")
    print(f"  Candidates : {len(candidates)}")

except (json.JSONDecodeError, ValueError) as e:
    print(f"✗ Director review parsing failed: {e}")
    print("Raw response:")
    print(review_response)
    raise RuntimeError(
        "Director review parsing failed — candidate set empty.\n"
        "Check the Director prompt or model output above."
    )

update_brief_doc(session_id, "DIRECTOR", "DIRECTOR REVIEW", review_response)
scribe_log(blackboard, "DIRECTOR", "review_complete",
           f"Director review complete. Decision: {'ITERATE' if review['iteration_required'] else 'PRESENT'}. "
           f"{len(blackboard['candidate_set'])} candidates selected.")

# Only validate candidate_set now if we're not about to iterate
# (Cell 8a will validate after iteration completes)
if not review["iteration_required"]:
    validate_stage(blackboard, "candidate_set")

print("── DIRECTOR REVIEW ─────────────────────────────────────────")
print(review_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Director review complete")
print(f"  Candidates selected : {len(blackboard['candidate_set'])}")
print(f"  Iteration required  : {review['iteration_required']}")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 8a — Stage 6a: Iteration Loop

Fires only when the Director signals ITERATE.
Re-runs the full Creative Team: Creator → Critic → Researcher.
Then re-runs Director review to produce a new candidate set.
Cell 9 (Presentation) proceeds normally from the updated candidate_set.

If ITERATE is not required, this cell prints a confirmation and passes through.

In [ ]:
# ── STAGE 6a: ITERATION LOOP ─────────────────────────────────────────────────

print("═" * 60)
print("STAGE 6a — ITERATION LOOP")
print("═" * 60)

if not review["iteration_required"]:
    print("✓ Director decision: PRESENT — no iteration needed")
    print(f"✓ Candidates in set: {len(blackboard['candidate_set'])}")

else:
    print("⚠ Director decision: ITERATE — re-running full Creative Team")
    print()

    # Clear existing ideation cycle and candidate set for a clean cycle 2
    blackboard["ideation_cycles"] = []
    blackboard["candidate_set"]   = []

    # ── CREATOR — CYCLE 2 ─────────────────────────────────────────────────────
    print("── CREATOR: CYCLE 2 ────────────────────────────────────────")

    iteration_context = (
        f"The Director reviewed the first ideation cycle and requested a full re-run.\n\n"
        f"DIRECTOR\'S FEEDBACK:\n{review_response}\n\n"
        f"BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Desired result: {brief['desired_result']}\n"
        f"Constraints: {', '.join(brief['constraints'])}\n\n"
        f"Generate 4 new distinct directions that address the Director\'s specific concerns above.\n"
        f"Use Verbalized Sampling: include probability scores. Include at least one below 0.15.\n\n"
        f"For each direction:\n"
        f"IDEA_ID: [A1, A2, A3, A4]\n"
        f"TITLE: [short evocative name]\n"
        f"PROBABILITY: [0.0-1.0]\n"
        f"CONCEPT: [2-3 sentences]\n"
        f"RATIONALE: [why this is worth exploring]\n"
        f"EMOTIONAL TERRITORY: [what feeling does this occupy?]"
    )

    creator_response_iter = call_role(
        role="creator",
        user_message=iteration_context,
        weights=ROLE_WEIGHTS["creator"],
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=1200,
        brief_doc=read_brief_doc(session_id)
    )

    cycle_2 = {
        "cycle_number": 2,
        "creator_proposals": [{"raw_response": creator_response_iter}],
        "critic_feedback": [],
        "synthesized_directions": []
    }
    blackboard["ideation_cycles"].append(cycle_2)
    update_brief_doc(session_id, "CREATOR", "DIRECTIONS — CYCLE 2", creator_response_iter)
    scribe_log(blackboard, "CREATOR", "iteration_ideation_complete",
               "Cycle 2 ideation complete following Director iteration request.")
    validate_stage(blackboard, "ideation")

    print(creator_response_iter)
    print()

    # ── CRITIC — CYCLE 2 ──────────────────────────────────────────────────────
    print("── CRITIC: CYCLE 2 ─────────────────────────────────────────")

    critic_task_iter = (
        f"The Creator has proposed a new set of directions following Director feedback.\n\n"
        f"BRIEF CHALLENGE: {brief['challenge']}\n"
        f"DESIRED RESULT: {brief['desired_result']}\n\n"
        f"CREATOR\'S NEW DIRECTIONS:\n{creator_response_iter}\n\n"
        f"Evaluate each direction. For each:\n"
        f"IDEA_ID: [matching the Creator\'s ID]\n"
        f"STRENGTHS: [what genuinely works]\n"
        f"WEAKNESSES: [what is underdeveloped or misaligned]\n"
        f"REFINEMENT: [one concrete suggestion]\n\n"
        f"Then propose ONE synthesis direction:\n"
        f"SYNTHESIS_ID: S1\n"
        f"TITLE: [name]\n"
        f"CONCEPT: [2-3 sentences]\n"
        f"ORIGIN_IDEAS: [which Creator ideas it draws from]\n\n"
        f"Be analytically rigorous."
    )

    critic_response_iter = call_role(
        role="critic",
        user_message=critic_task_iter,
        weights=ROLE_WEIGHTS["critic"],
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=1800,
        brief_doc=read_brief_doc(session_id)
    )

    blackboard["ideation_cycles"][-1]["critic_feedback"].append(
        {"raw_response": critic_response_iter}
    )
    update_brief_doc(session_id, "CRITIC", "CRITIQUE — CYCLE 2", critic_response_iter)
    scribe_log(blackboard, "CRITIC", "iteration_evaluation_complete",
               "Cycle 2 Critic evaluation complete.")
    validate_stage(blackboard, "critic")

    print(critic_response_iter)
    print()

    # ── RESEARCHER — CYCLE 2 ──────────────────────────────────────────────────
    print("── RESEARCHER: CYCLE 2 ─────────────────────────────────────")

    research_task_iter = (
        f"You are supporting a creative studio session. The Creative Team has completed "
        f"a second ideation cycle.\n\n"
        f"THE BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Desired result: {brief['desired_result']}\n\n"
        f"CREATOR\'S DIRECTIONS (CYCLE 2):\n{creator_response_iter}\n\n"
        f"CRITIC\'S EVALUATION (CYCLE 2):\n{critic_response_iter}\n\n"
        f"Your task: source and cite 2-3 external precedents, examples, or factual findings\n"
        f"from outside this problem\'s domain that could ground or expand what the Creative\n"
        f"Team has produced. Respond to the actual directions above.\n\n"
        f"For each finding:\n"
        f"TOPIC: [what area, domain, or precedent]\n"
        f"SOURCE: [name the specific source, study, institution, or example — be precise]\n"
        f"FINDING: [what it shows]\n"
        f"RELEVANCE: [why this matters for these specific directions]\n\n"
        f"If you have no external knowledge that would genuinely add here, say only:\n"
        f"RESEARCHER: No external contribution warranted for this cycle.\n\n"
        f"You are a sourcing agent, not an analyst. Do not evaluate the Creative Team\'s work."
    )

    research_response_iter = call_role(
        role="researcher",
        user_message=research_task_iter,
        weights=ROLE_WEIGHTS["researcher"],
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=700,
        brief_doc=read_brief_doc(session_id)
    )

    research_entry_iter = {
        "research_id":           f"R{len(blackboard['research_trace']) + 1:03d}",
        "initiated_by":          "director_iteration",
        "query":                 "Post-ideation cycle 2 enrichment",
        "sources":               ["researcher_knowledge"],
        "summary":               research_response_iter,
        "influence_on_ideation": "Cycle 2 post-iteration research"
    }
    blackboard["research_trace"].append(research_entry_iter)
    brief["research_insights"].append(research_response_iter)
    update_brief_doc(session_id, "RESEARCHER", "RESEARCH — CYCLE 2", research_response_iter)
    scribe_log(blackboard, "RESEARCHER", "iteration_research_delivered",
               f"Cycle 2 research. Entry R{len(blackboard['research_trace']):03d} logged.")

    print(research_response_iter)
    print()

    # ── DIRECTOR REVIEW — CYCLE 2 ─────────────────────────────────────────────
    print("── DIRECTOR REVIEW: CYCLE 2 ────────────────────────────────")

    review_task_iter = (
        f"You are reviewing the studio\'s second ideation cycle before presenting to the PIL.\n\n"
        f"BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Desired result: {brief['desired_result']}\n"
        f"Constraints: {', '.join(brief['constraints'])}\n\n"
        f"CREATOR DIRECTIONS (CYCLE 2):\n{creator_response_iter[:800]}\n\n"
        f"CRITIC EVALUATION (CYCLE 2):\n{critic_response_iter[:800]}\n\n"
        f"Select 3 directions to present. Return ONLY a JSON object — "
        f"no preamble, no markdown fences:\n\n"
        f"{{\n"
        f"  \"candidates\": [\n"
        f"    {{\"id\": \"C1\", \"title\": \"...\", \"type\": \"evolutionary|evolutionary_variant|conceptual_leap|sacrificial_probe\", \"source_idea_id\": \"A1\"}},\n"
        f"    {{\"id\": \"C2\", \"title\": \"...\", \"type\": \"...\", \"source_idea_id\": \"A2\"}},\n"
        f"    {{\"id\": \"C3\", \"title\": \"...\", \"type\": \"...\", \"source_idea_id\": \"S1\"}}\n"
        f"  ],\n"
        f"  \"alignment\": \"...\",\n"
        f"  \"distinctiveness\": \"...\",\n"
        f"  \"balance\": \"...\",\n"
        f"  \"clarity\": \"...\",\n"
        f"  \"decision\": \"PRESENT\",\n"
        f"  \"reason\": \"...\"\n"
        f"}}"
    )

    review_response_iter = call_role(
        role="director",
        user_message=review_task_iter,
        weights=ROLE_WEIGHTS["director"],
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=800,
        brief_doc=read_brief_doc(session_id)
    )

    # Parse cycle 2 Director review
    try:
        clean_iter = review_response_iter.strip()
        if clean_iter.startswith("```"):
            clean_iter = clean_iter.split("```")[1]
            if clean_iter.startswith("json"):
                clean_iter = clean_iter[4:]
        parsed_iter = json.loads(clean_iter.strip())

        review["alignment_with_brief"]       = parsed_iter.get("alignment", "")
        review["distinctiveness_assessment"] = parsed_iter.get("distinctiveness", "")
        review["balance_assessment"]         = parsed_iter.get("balance", "")
        review["clarity_assessment"]         = parsed_iter.get("clarity", "")
        review["iteration_required"]         = False  # reset after cycle 2 review

        for c in parsed_iter.get("candidates", []):
            blackboard["candidate_set"].append({
                "direction_id":      c.get("id", f"D{len(blackboard['candidate_set'])+1:03d}"),
                "internal_type":     c.get("type", "evolutionary").lower(),
                "description":       c.get("title", ""),
                "reasoning_summary": "",
                "research_influences": [],
                "critic_assessment": {"strengths": [], "concerns": [], "refinement_notes": []}
            })

        print("✓ Cycle 2 Director review parsed successfully")

    except (json.JSONDecodeError, ValueError) as e:
        print(f"✗ Cycle 2 Director review parsing failed: {e}")
        print(review_response_iter)
        raise RuntimeError(
            "Iteration Director review failed — candidate set still empty.\n"
            "Check the Director prompt or model output above."
        )

    update_brief_doc(session_id, "DIRECTOR", "DIRECTOR REVIEW — CYCLE 2", review_response_iter)
    scribe_log(blackboard, "DIRECTOR", "iteration_review_complete",
               f"Cycle 2 Director review complete. {len(blackboard['candidate_set'])} candidates selected.")
    validate_stage(blackboard, "candidate_set")

    # Update creator_output and critic_output so downstream cells (9 and 11)
    # read from cycle 2, not cycle 1.
    creator_output = creator_response_iter
    critic_output  = critic_response_iter

    print()
    print(f"✓ Iteration complete")
    print(f"✓ Candidates in set : {len(blackboard['candidate_set'])}")
    print(f"✓ Reasoning trace   : {len(blackboard['reasoning_trace'])} entries")

---
## Cell 9 — Stage 7: Presentation

Director presents the candidate directions to the PIL.
Each direction includes reasoning. Tone is clear, professional, non-salesy.

In [ ]:
# ── STAGE 7: PRESENTATION ────────────────────────────────────────────────────

print("═" * 60)
print("STAGE 7 — PRESENTATION")
print("═" * 60)

candidate_summary = "\n".join([
    f"- {c['description']} (type: {c['internal_type']})"
    for c in blackboard["candidate_set"]
])

presentation_task = f"""Present these three directions to the person in the loop.

BRIEF CHALLENGE: {brief['challenge']}

CANDIDATE DIRECTIONS:
{candidate_summary}

FULL STUDIO OUTPUT TO DRAW FROM:
Creator: {creator_output[:600]}
Critic: {critic_output[:600]}

Write a clear, engaging presentation. For each direction:
- Give it a memorable name
- Describe it in 2-3 sentences the PIL can actually picture
- Explain briefly why it deserves consideration

Sequence matters: lead with the strongest, end with the most unexpected.
Tone: warm, direct, confident. No hype. No 'exciting' or 'powerful'.
Close with a single question that invites reaction."""

presentation_response = call_role(
    role="director",
    user_message=presentation_task,
    weights=ROLE_WEIGHTS["director"],
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=1000,
    brief_doc=read_brief_doc(session_id)
)

# Store presentation
blackboard["presentation"]["ordered_directions"] = [
    c["direction_id"] for c in blackboard["candidate_set"]
]
blackboard["presentation"]["director_commentary"].append(presentation_response)

scribe_log(blackboard, "DIRECTOR", "presentation_delivered",
           f"Presentation delivered to PIL. {len(blackboard['candidate_set'])} directions presented.")

print("── DIRECTOR PRESENTS TO PIL ────────────────────────────────")
print(presentation_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Presentation complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 10 — Stage 8: User Reaction

PIL responds to the candidate directions.
Director extracts creative signals from the reaction.
These signals determine whether to proceed to synthesis or trigger a second loop.

In [ ]:
# ── STAGE 8: USER REACTION ────────────────────────────────────────────────────

print("═" * 60)
print("STAGE 8 — USER REACTION")
print("═" * 60)

if TEST_MODE:
    # Simulated PIL reaction for pipeline testing
    pil_reaction = (
        "I really like the third direction — the unexpected one. "
        "The first two feel a bit expected honestly. "
        "Can we push the third one further? I want something that feels "
        "more like a manifesto than a brand. "
        "The belonging idea from earlier keeps coming back to me."
    )
    print(f"[TEST MODE — simulated reaction]")
    print(f"\nPIL: {pil_reaction}\n")
else:
    pil_reaction = input("Your reaction: ").strip()

# Director extracts signals from reaction
signal_task = f"""The PIL responded to the presentation:
"{pil_reaction}"

Extract the creative signals. Identify:
SELECTED: [which direction(s) resonated, if any]
SIGNAL_1: [key preference or instinct revealed]
SIGNAL_2: [what they want more or less of]
SIGNAL_3: [any new insight or direction revealed]
SECOND_LOOP: YES | NO
REASON: [why second loop is or isn't needed]"""

signal_response = call_role(
    role="director",
    user_message=signal_task,
    weights=ROLE_WEIGHTS["director"],
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=400,
    brief_doc=read_brief_doc(session_id)
)

# Store user response
user_resp = blackboard["user_response"]
user_resp["feedback_summary"] = pil_reaction
user_resp["signals_extracted"].append(signal_response)

# Determine if second loop needed
second_loop_needed = "SECOND_LOOP: YES" in signal_response.upper()
blackboard["second_exploration"]["triggered"] = second_loop_needed
if second_loop_needed:
    blackboard["second_exploration"]["reason"] = "PIL reaction indicates direction needs further development"

# Update Studio Brief Document
update_brief_doc(session_id, "DIRECTOR", "PIL SIGNALS",
                 f"**PIL reaction:** {pil_reaction}\n\n**Signals extracted:**\n{signal_response}")

scribe_log(blackboard, "DIRECTOR", "signals_extracted",
           f"PIL reaction processed. Second loop: {second_loop_needed}. "
           f"Key signal: '{pil_reaction[:60]}...'")

print("── CREATIVE SIGNALS EXTRACTED ──────────────────────────────")
print(signal_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Signals extracted")
print(f"  Second loop triggered : {second_loop_needed}")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 10b — Stage 8b: Depth Extraction

Director uses one extraction technique from the toolkit to probe deeper
before committing to final synthesis.
In TEST_MODE: PIL depth response is simulated.
In live mode: Director asks one question, PIL responds.

In [ ]:
# ── STAGE 8b: DEPTH EXTRACTION ───────────────────────────────────────────────

print("═" * 60)
print("STAGE 8b — DEPTH EXTRACTION")
print("═" * 60)

depth_task = (
    f"The PIL has reacted to the presentation. Before producing the final synthesis,\n"
    f"probe one level deeper using a single extraction technique from your toolkit.\n\n"
    f"PIL REACTION:\n{pil_reaction}\n\n"
    f"SIGNALS EXTRACTED SO FAR:\n{signal_response}\n\n"
    f"Choose ONE technique that would surface the most useful additional signal right now.\n"
    f"Ask a single focused question — not multiple questions.\n"
    f"Keep it conversational. Do not explain why you\'re asking. Just ask."
)

depth_question = call_role(
    role="director",
    user_message=depth_task,
    weights=ROLE_WEIGHTS["director"],
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=150,
    brief_doc=read_brief_doc(session_id)
)

print("── DIRECTOR DEPTH QUESTION ─────────────────────────────────")
print(depth_question)
print()

if TEST_MODE:
    depth_response = (
        "The belonging piece is central — I want people to feel like they helped build "
        "this place, not just that they\'re welcome in it. Like they\'d be missed if they "
        "stopped coming."
    )
    print("[TEST MODE — simulated depth response]")
    print(f"PIL: {depth_response}")
else:
    depth_response = input("> ").strip()

# Store depth signal alongside Stage 8 signals
blackboard["user_response"]["signals_extracted"].append(
    f"DEPTH SIGNAL (Stage 8b): {depth_response}"
)
update_brief_doc(session_id, "DIRECTOR", "DEPTH SIGNAL",
                 f"**Director question:** {depth_question}\n\n"
                 f"**PIL response:** {depth_response}")
scribe_log(blackboard, "DIRECTOR", "depth_extraction_complete",
           f"Stage 8b depth extraction complete. Signal: '{depth_response[:80]}'")

print()
print("✓ Depth signal captured")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 11 — Stage 9: Final Synthesis

Director produces the final synthesis — refining the chosen direction
and incorporating signals from the PIL's reaction.
This is the studio's best work.

In [ ]:
# ── STAGE 9: FINAL SYNTHESIS ─────────────────────────────────────────────────

print("═" * 60)
print("STAGE 9 — FINAL SYNTHESIS")
print("═" * 60)

all_signals = "\n\n".join(blackboard["user_response"]["signals_extracted"])
signals = all_signals if all_signals else ""

synthesis_task = f"""Produce the final synthesis for the PIL.

BRIEF: {brief['challenge']}
DESIRED RESULT: {brief['desired_result']}

PIL REACTION: {pil_reaction}

SIGNALS EXTRACTED:
{signals}

STUDIO DIRECTIONS EXPLORED:
{creator_output[:400]}

Synthesize the most compelling direction — refined by the PIL's signals.
This may combine elements from multiple directions.

Structure:
FINAL DIRECTION: [name]
CORE CONCEPT: [3-4 sentences — the fully developed idea]
WHY THIS WORKS: [2-3 sentences connecting it to the brief and PIL's signals]
NEXT STEPS: [2-3 concrete actions the PIL could take to move forward]

Tone: confident, clear, earned. This is the studio's best thinking."""

synthesis_response = call_role(
    role="director",
    user_message=synthesis_task,
    weights=ROLE_WEIGHTS["director"],
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=800,
    brief_doc=read_brief_doc(session_id)
)

# Store final synthesis
synthesis = blackboard["final_synthesis"]
synthesis["summary"] = synthesis_response
if blackboard["candidate_set"]:
    synthesis["final_direction"] = blackboard["candidate_set"][-1]["direction_id"]
synthesis["refinements"].append(f"Refined from PIL signal: '{pil_reaction[:100]}'")

scribe_log(blackboard, "DIRECTOR", "synthesis_complete",
           "Final synthesis delivered to PIL. Session complete.")

print("── FINAL SYNTHESIS ─────────────────────────────────────────")
print(synthesis_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Final synthesis complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

---
## Cell 12 — Session Record: Blackboard Inspection + Save

Complete session record. Full reasoning trace visible.
Session saved to JSON — raw material for Phase 3 visualization.

In [ ]:
# ── SESSION RECORD ────────────────────────────────────────────────────────────

print("═" * 60)
print("SESSION RECORD")
print("═" * 60)
print(f"Session ID      : {blackboard['session_metadata']['session_id']}")
print(f"Prompt          : {blackboard['user_prompt']}")
print(f"Test mode       : {TEST_MODE}")
print(f"Model           : {SESSION_MODEL}")
print()
print(f"Ideation cycles : {len(blackboard['ideation_cycles'])}")
print(f"Research entries: {len(blackboard['research_trace'])}")
print(f"Candidate dirs  : {len(blackboard['candidate_set'])}")
print(f"Reasoning steps : {len(blackboard['reasoning_trace'])}")
print()
print("── REASONING TRACE ─────────────────────────────────────────")
for entry in blackboard["reasoning_trace"]:
    print(f"  {entry['step']:>2} | {entry['role']:<12} | {entry['action']:<22} | {entry['summary'][:60]}")

print()
print("── CANDIDATE DIRECTIONS ────────────────────────────────────")
for c in blackboard["candidate_set"]:
    print(f"  {c['direction_id']} | {c['internal_type']:<22} | {c['description']}")

# Save full session
saved_path = save_session(blackboard)
print()
print(f"✓ Full session saved to: {saved_path}")
print()
print("Phase 1 complete. Full studio workflow executed.")
print()
print("This JSON is Phase 3 input — the semantic trajectory data.")
print("Every reasoning_trace entry is an utterance in conceptual space.")